# 27c · `dataset_v3` cho 4 arm multi-view — bản sẵn dùng

> **Người nhận:** mục **0–2** là đủ để nạp dữ liệu và train · **3** bắt buộc đọc nếu dùng
> arm `frontal` · **4–5** là những chỗ dễ vấp · **6** dành cho người chạy notebook.

Notebook này lấy `dataset_v3` (do `07` sinh, pooled `0.7/0.3`) và đẻ ra **một thư mục cùng
cấu trúc cho mỗi tỉ lệ pooling**. Bên nhận chỉ việc trỏ model vào thư mục tương ứng — không
cần tự pool, không cần chạy backbone, không cần code của repo này.

```
planB_out/
├── dataset_v3/           <- ĐÃ CÓ (07). Dùng làm cổng đối chiếu, không đụng vào.
├── dataset_v3_73/        <- 0.7 / 0.3   (dựng lại, phải khớp dataset_v3)
├── dataset_v3_64/        <- 0.6 / 0.4
├── dataset_v3_55/        <- 0.5 / 0.5
└── dataset_v3_frontal/   <- chỉ frontal  (đọc mục 3 — arm duy nhất có ca biên)
```

`dataset_v3_73` là bản dựng lại của `dataset_v3`, không phải bản sao. Có nó thì bốn arm
**đối xứng** — bên nhận đổi arm bằng cách đổi tên thư mục, không phải nhớ "à cái 7:3 thì
tên khác". Và việc dựng lại chính là **cổng mạnh nhất** của notebook: nếu nó khớp
`dataset_v3` thì thứ tự manifest, công thức trọng số và cách chuẩn hoá đều đúng, cùng lúc.

**Arm `attention` để sau.** Nó không cùng dạng được với bốn arm này — trọng số attention do
model sinh lúc forward, nên **không tồn tại `x_cxr` tĩnh** cho nó; thư mục của nó sẽ phải
chứa bảng per-image + chỉ mục ragged thay vì một ma trận `[N, 768]`. Xem mục cuối.

---

## 0 · Mỗi thư mục có gì

Giống hệt `dataset_v3` của `07` — cùng tên file, cùng schema, cùng thứ tự hàng:

| file | shape | nội dung |
|---|---|---|
| `v1_onehot.npy` | `[211130, 1272]` | `micro_onehot(475) ⊕ ehr(29) ⊕ x_cxr(768)` |
| `v2_embedding.npy` | `[211130, 829]` | `micro_embed(32) ⊕ ehr(29) ⊕ x_cxr(768)` |
| `index.parquet` | `[211130, 18]` | `study_id`, `split`, `setting`, `row`, **14 nhãn** |
| `cohort_study_level.parquet` | `[211130, …]` | bảng cohort đầy đủ (EHR + 14 nhãn) **⊕ `x_cxr` của arm này** |
| `class_freq.pkl` | — | `labels`/`class_freq`/`num_train` cho **14** nhãn (cái ở `cohort_v2/` chỉ có 13) |
| `columns.json` | — | tên từng cột theo đúng thứ tự, nguồn, dtype |
| `x_cxr_train_stats.npz` | — | `mean`/`std` train-only **của chính arm đó** |
| `v*_sample2000.csv` | — | mẫu nhỏ để liếc schema |

`v1` và `v2` **chứa cùng một lượng thông tin** — chúng chỉ khác cách mã hoá khối
microbiology (one-hot vs embedding), và `07` đã chứng minh đó là một ánh xạ tuyến tính
chính xác (`onehot @ W == EmbeddingBag`). Dùng `v2` cho mạng nơ-ron, `v1` cho cây.

**Chỉ 768 cột `x_cxr` khác nhau giữa các arm.** Khối micro, khối EHR, nhãn, thứ tự hàng
— bit-identical, `C5` kiểm chứng. Nghĩa là mọi Δ đo được quy về pooling, không lẫn thứ gì
khác. (`index.parquet` của arm `frontal` có thêm đúng một cột `frontal_is_fallback` —
xem mục 3.)

---

## 1 · Nạp và train

```python
import numpy as np, pandas as pd, json

D    = 'dataset_v3_64'                          # <- doi arm = doi dong nay
idx  = pd.read_parquet(f'{D}/index.parquet')
X    = np.load(f'{D}/v2_embedding.npy')
spec = json.load(open(f'{D}/columns.json'))

tr = idx.split == 'train'
Xtr, ytr = X[idx.row[tr]], idx.loc[tr, spec['labels']].to_numpy()
```

Đổi arm = **đổi tên thư mục**. Không đổi một dòng nào khác.

---

## 2 · Chỉ dùng riêng khối ảnh

`x_cxr` luôn là 768 cột **cuối cùng**:

```python
d = spec['x_cxr']['dim']                        # 768
x_cxr  = X[:, -d:]                              # chi khoi anh -- KHAC nhau giua cac arm
static = X[:, :-d]                              # micro + ehr -- GIONG HET o moi arm
```

Muốn ablation thuần ảnh thì train trên `x_cxr` thôi; muốn fusion thì dùng cả `X`.

---

## 3 · ⚠ Arm `frontal` và 366 study chỉ có lateral

Ba arm tỉ lệ đều có `w_f, w_l > 0` nên mẫu số tái chuẩn hoá `w_f·[∃F] + w_l·[∃L]` luôn
dương. Arm `frontal` (`w_f=1.0`) là arm **duy nhất** chạm ca biên: với 366 study không có
ảnh frontal nào, mẫu số **bằng 0**.

`04b` xử lý bằng cách *loại* chúng. Notebook này **không** làm thế, vì như vậy arm `frontal`
sẽ được chấm trên tập study khác ba arm kia, và Δ không còn quy về pooling nữa — nó lẫn cả
"đổi cohort". Thay vào đó: các study đó dùng **lateral mean** ("dùng cái đang có"), nên

- **`N` giống hệt cả bốn arm** → Δ giữa các arm là Δ thuần pooling;
- cột **`frontal_is_fallback`** trong `index.parquet` của **riêng** arm này đánh dấu chính
  xác 366 study đó.

**Muốn theo quy ước `04b`:** lọc `frontal_is_fallback` ở **downstream, trên cả 4 arm**:

```python
fb  = pd.read_parquet('dataset_v3_frontal/index.parquet').frontal_is_fallback.to_numpy()
keep = ~fb                                      # dung MOT mask nay cho MOI arm
Xtr = X[idx.row[tr & keep]]
```

Lọc chỉ ở arm `frontal` là đúng cái sai mà thiết kế này tránh — 0.17% cohort, nhưng nó biến
một ablation sạch thành một phép so hai thứ khác nhau.

---

## 4 · Chuẩn hoá — đọc trước khi so arm

Mỗi arm mang `x_cxr_train_stats.npz` **của riêng nó** (mean/std trên study `train` của
chính arm đó), và `x_cxr` trong file `.npy` đã được áp stats đó rồi.

Vì sao per-arm chứ không dùng chung stats của `0.7/0.3`: pooling càng nhiều ảnh thì phương
sai càng co, nên dùng chung stats sẽ khiến các arm tới model với `std` khác nhau — tức khác
**learning rate hiệu dụng**, một confound thuần kỹ thuật chẳng liên quan gì tới pooling.
Per-arm thì mọi arm đều tới ở `mean 0 / std 1`, khác biệt còn lại đúng là *cấu trúc* của
vector.

Với arm `73`, stats per-arm **chính là** `x_cxr_train_stats.npz` của `05` — `C3` dùng đúng
điều đó làm cổng.

---

## 5 · Ba chỗ dễ vấp

1. **Chia split theo `subject_id`, không theo study.** Cột `split` có sẵn đã làm đúng; nếu
   chia lại mà quên ràng buộc thì ảnh cùng bệnh nhân rơi vào cả train lẫn test.
2. **`NaN` chỉ có trong khối EHR, và là cố ý.** `07` bỏ median-impute để cây (XGBoost) đọc
   `NaN` trực tiếp — giữ đúng phân phối thật thay vì một cột dựng đứng ở median. Micro và
   `x_cxr` **không bao giờ** `NaN`; `C5` assert điều đó. Mạng nơ-ron thì phải tự impute
   lại, hoặc chạy `07` với `BLANK_IMPUTED=False`.
3. **Đừng so arm bằng một seed.** Chênh lệch giữa các tỉ lệ pooling ở bộ này rất nhỏ. Chạy
   ≥ 3 seed và so Δ với độ lệch chuẩn giữa các seed *trước* — Δ nhỏ hơn nhiễu train thì
   không phải phát hiện.

### Bối cảnh, để đặt kỳ vọng

Đo ở **mức xác suất** (model đóng băng, chỉ đổi cách trộn đầu ra), pooling học được từng
cho `dMacro +0.00047, CI95 [-0.00096, +0.00199]` — **chưa loại 0**. Bộ dataset này phục vụ
câu hỏi ở **mức embedding** (head train lại), khác và chưa ai đo — nhưng con số trên nói
rằng dư địa ở trục này rất mỏng. Đừng kỳ vọng một cú nhảy.

---

## 6 · Chạy notebook này

**Cần:** `07` đã chạy (`dataset_v3/`), `05` đã chạy (`cxr_cls.npy` — cache **per-image**,
không phải `study_x_cxr.npy`), `03` đã chạy (`view_map.parquet`).

**Phần cứng:** CPU thuần, **không cần GPU**. RAM đỉnh ~3.5 GB. Thời gian ~15–25 phút, phần
lớn là ghi xuống Drive.

**Dung lượng: ~7.1 GB** (mỗi arm 1.77 GB = `v1` 1.07 + `v2` 0.70, bốn arm). `C0` in con số thật
trước khi ghi. Chỉ cần một variant thì sửa `VARIANTS` ở `C0`.

| cell | làm gì | cổng |
|---|---|---|
| `C0` | config, kiểm prereq, ước lượng dung lượng | — |
| `C1` | nạp `dataset_v3`, định vị khối `x_cxr` bằng `columns.json` | 768 cột cuối phải tên `x_cxr_*` |
| `C2` | manifest ảnh + CLS per-image | mọi dicom trong cache có trong `view_map` |
| `C3` | trọng số từng arm | `sum(w)=1` mọi arm; **`73` tái lập `x_cxr` của `dataset_v3`**; fallback `frontal` đúng 366 study |
| `C4` | ghi 4 thư mục | không `NaN` trong `x_cxr`; số cột khớp `columns.json` |
| `C5` | nghiệm thu, nạp lại từ đĩa | non-image **bit-identical**; `x_cxr` phải **khác** nhau; `_73` ≈ `dataset_v3` |
| `C6` | liệt kê + cách nạp | — |

---

*Notebook này thuộc nhánh `cohort_v2` (01–27), không phải nhánh CXR-LT (28+).*

## C0 · Config

In [ ]:
import os, json, time, pickle
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('PLANB_OUT', './planB_out')

OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
CACHE_DIR = os.path.join(DATA_DIR, 'cxr_cls_cache')
SRC_DS    = os.path.join(DATA_DIR, 'dataset_v3')      # do 07 sinh, pooled 0.7/0.3

# thu muc -> w_f.  w_f=1.0 la arm chi-frontal (xem muc 3 o header: 366 study lateral-only).
ARMS = {
    'dataset_v3_73':      0.7,
    'dataset_v3_64':      0.6,
    'dataset_v3_55':      0.5,
    'dataset_v3_frontal': 1.0,
}
VARIANTS         = ('v1_onehot', 'v2_embedding')   # nhu dataset_v3. Bo bot de tiet kiem dia.
WRITE_CSV_SAMPLE = 2000     # CHI cat file CSV xem-bang-mat. .npy LUON du N hang. None = bo CSV.
WRITE_COHORT_PARQUET = True  # moi arm mot cohort_study_level.parquet = cohort + x_cxr cua arm do
COHORT_PARQUET_WITH_XCXR = True   # False -> chi copy bang cohort (giong nhau moi arm)
FORCE_REWRITE_NPY = False    # False: .npy da co thi BO QUA (khong ghi lai 7.2 GB)

# ---- nhan ----
# 01 co chu y bo 'Support Devices' (DROP_LABELS) voi ly do "not a pathology (tubes/lines)".
# Tuan yeu cau lay lai -> 14 nhan. Lay tu DUNG nguon cua 01 va DUNG quy uoc U-Ones.
ADD_SUPPORT_DEVICES = True
EXPECT_N_LABELS     = 14      # 13 (cohort_v2) + Support_Devices. TUYET DOI khong phai 40 (CXR-LT).
PROJECT = 'mimic-ehr-v310'                  # nhu 01 C0
CXR     = 'physionet-data.mimic_cxr_jpg'    # nhu 01 C0 -> bang {CXR}.chexpert
U_POLICY = 'ones'                           # nhu 01 C0: -1 -> 1, NaN -> 0
# Neu khong muon dung BigQuery: tro vao mot file co 2 cot study_id + Support_Devices (0/1)
SUPPORT_DEVICES_SRC = os.environ.get('SD_SRC')   # None -> lay tu BigQuery nhu 01

DTYPE            = np.float32

assert os.path.isdir(SRC_DS), f'{SRC_DS} khong co -> chay 07_export_dataset.ipynb truoc'
for _f in ['index.parquet', 'columns.json']:
    assert os.path.exists(os.path.join(SRC_DS, _f)), f'{_f} thieu trong {SRC_DS}'
for _f in ['cxr_cls.npy', 'cxr_cls_index.parquet']:
    assert os.path.exists(os.path.join(CACHE_DIR, _f)), \
        f'{_f} khong co -> chay 05 (can cache PER-IMAGE, khong phai study_x_cxr.npy)'
assert os.path.exists(os.path.join(OUT_DIR, 'view_map.parquet')), 'view_map.parquet -> chay 03'
COHORT_SRC = os.path.join(OUT_DIR, 'cohort_study_level.parquet')
if WRITE_COHORT_PARQUET:
    assert os.path.exists(COHORT_SRC), f'{COHORT_SRC} khong co -> chay 02'

_sp = json.load(open(os.path.join(SRC_DS, 'columns.json')))
_N  = int(_sp['n_studies'])
for _v in VARIANTS:
    assert _v in _sp['variants'], f'{_v} khong co trong columns.json cua 07'
_per_arm = sum(_N * len(_sp['variants'][v]['columns']) * 4 for v in VARIANTS) / 1e9

print(f'nguon : {SRC_DS}   ({_N:,} studies, x_cxr {_sp["x_cxr"]["dim"]}-d)')
print(f'arms  : {list(ARMS)}')
print(f'ghi   : {[os.path.join(DATA_DIR, d) for d in ARMS]}')
print(f'variants: {list(VARIANTS)}')
print(f'\nSE GHI ~{_per_arm*len(ARMS):.1f} GB  ({_per_arm:.2f} GB/arm x {len(ARMS)} arm)')
print('  -> bo bot VARIANTS neu muon nhe hon')

## C1 · Nạp `dataset_v3`, tách khối `x_cxr`

Chỗ dễ hỏng nhất là **giả định `x_cxr` nằm ở 768 cột cuối**. Thay vì tin, cell này đọc tên
cột từ `columns.json` và *kiểm chứng* — nếu `07` từng đổi thứ tự block thì nó dừng ở đây
chứ không lặng lẽ ghi ra dataset lệch cột.

Khối còn lại (`micro ⊕ ehr`) giữ nguyên để dùng chung cho mọi arm. Nó chỉ chiếm phần nhỏ
nên lặp ở mỗi thư mục là cái giá rẻ để mỗi arm **tự chứa**.

In [ ]:
SPEC0    = json.load(open(os.path.join(SRC_DS, 'columns.json')))
LABELS13 = list(SPEC0['labels'])
assert len(LABELS13) != 40, ('columns.json co 40 nhan -> day la nhanh CXR-LT (28+), '
                             'KHONG phai cohort_v2. Sai SRC_DS.')
assert len(LABELS13) == 13, f'dataset_v3 co {len(LABELS13)} nhan, mong doi 13 (cohort_v2)'
LABELS = LABELS13 + (['Support_Devices'] if ADD_SUPPORT_DEVICES else [])
C = len(LABELS)
assert C == EXPECT_N_LABELS, f'{C} nhan, mong doi {EXPECT_N_LABELS}'
print(f'nhan: {C}  (13 tu dataset_v3'
      + (' + Support_Devices lay lai o C1-bis)' if ADD_SUPPORT_DEVICES else ')'))
for i, l in enumerate(LABELS):
    print(f'  {i+1:2d}. {l}' + ('   <- THEM MOI' if l not in LABELS13 else ''))
IDX0   = pd.read_parquet(os.path.join(SRC_DS, 'index.parquet'))
N      = len(IDX0); assert N == SPEC0['n_studies']
DX     = int(SPEC0['x_cxr']['dim'])
SID    = IDX0.study_id.to_numpy()
assert (IDX0.row.to_numpy() == np.arange(N)).all(), 'index.parquet: row phai la 0..N-1'
assert IDX0.study_id.is_monotonic_increasing, 'index.parquet phai sort theo study_id'

STATIC = {}
for v in VARIANTS:
    cols = SPEC0['variants'][v]['columns']
    p    = os.path.join(SRC_DS, SPEC0['variants'][v]['file'])
    assert os.path.exists(p), f'{p} khong co -> chay lai 07'
    tail = cols[-DX:]
    assert tail == [f'x_cxr_{i}' for i in range(DX)], \
        f'{v}: {DX} cot cuoi khong phai x_cxr_* -> 07 da doi thu tu block, dung lai'
    assert not any(c.startswith('x_cxr_') for c in cols[:-DX]), f'{v}: x_cxr lan ra ngoai khoi cuoi'
    V = np.load(p, mmap_mode='r')
    assert V.shape == (N, len(cols)), (v, V.shape, len(cols))
    STATIC[v] = np.ascontiguousarray(V[:, :-DX]).astype(DTYPE)
    if v == VARIANTS[0]:
        XREF = np.ascontiguousarray(V[:, -DX:]).astype(DTYPE)   # x_cxr cua 0.7/0.3, cho cong C3
    del V
    print(f'{v:14s} {len(cols):5d} cot = static {STATIC[v].shape[1]:4d} + x_cxr {DX}'
          f'   static {STATIC[v].nbytes/1e6:6.0f} MB')

assert not np.isnan(XREF).any(), 'x_cxr nguon co NaN -> khong the'
IS_TRAIN = (IDX0.split.to_numpy() == 'train')

COHORT = None
if WRITE_COHORT_PARQUET:
    COHORT = pd.read_parquet(COHORT_SRC).sort_values('study_id').reset_index(drop=True)
    assert len(COHORT) == N and (COHORT.study_id.to_numpy() == SID).all(),         'cohort_study_level.parquet lech voi index.parquet cua dataset_v3 -> khac cohort'
    _miss = [l for l in LABELS13 if l not in COHORT.columns]
    assert not _miss, f'cohort nguon thieu nhan: {_miss}'
    assert not any(c.startswith('x_cxr_') for c in COHORT.columns), 'cohort nguon da co x_cxr_*'
    print(f'cohort nguon: {COHORT.shape[0]:,} hang x {COHORT.shape[1]} cot')
print(f'\nstudies {N:,} | nhan {C} | split', dict(IDX0.split.value_counts()))

## C1-bis · Lấy lại nhãn thứ 14 — `Support_Devices`

`01` **cố ý** bỏ nhãn này (`DROP_LABELS = ['Support Devices']`), lý do ghi trong bảng
deviation của nó: *"not a pathology (tubes/lines/pacemakers)"*. Lấy lại được, nhưng phải
lấy từ **đúng nguồn** và **đúng quy ước** của `01`, không thì nhãn 14 sống theo luật khác
13 nhãn kia:

| | `01` làm gì | ở đây |
|---|---|---|
| nguồn | BigQuery `physionet-data.mimic_cxr_jpg.chexpert` | y hệt |
| U-policy | `U_POLICY='ones'` → `-1 → 1` | y hệt |
| thiếu | `fillna(0)`, `int8` | y hệt |
| join | LEFT trên `study_id`, không được rớt study nào | y hệt |
| tên cột | đọc **từ schema**, không đoán | y hệt |

**Split không đổi.** `01` sinh split bằng `stratified_group_split` trên **13** nhãn; nhãn
14 chỉ được *ghép thêm*, cột `split` giữ nguyên. Không tái sinh split — làm thế sẽ phá
tính so sánh được với `dataset_v3` và với mọi kết quả cũ.

### ⚠ Nhãn này là một confound đã biết

`Support Devices` không phải bệnh lý — nó là ống, dây, máy tạo nhịp. Nó tương quan mạnh
với "bệnh nhân nặng / nằm ICU", nên model rất dễ dùng nó làm đường tắt và điểm số ở các
nhãn khác trông đẹp lên mà không phải vì đọc được phổi. Đó chính là lý do `01` bỏ nó. Giữ
nó thì nên **báo cáo riêng**, đừng gộp vào macro chung rồi so với số cũ 13 nhãn.

### Auth

Colab **không** có service account mặc định. Bỏ qua bước auth thì `bigquery.Client()` rơi
xuống credential của GCE metadata và chết bằng `RefreshError: ... metadata.google.internal
... Status: 404` — không phải lỗi quyền, chỉ là chưa có credential nào.

Cell `gcloud` dưới in ra một URL: mở → đăng nhập → copy mã → dán vào ô input. **Không**
dùng `google.colab.auth.authenticate_user()`. Cell kế tiếp ép dùng đúng file ADC vừa tạo
và dừng bằng câu tiếng người nếu chưa auth.

*(Không muốn đụng BigQuery: đặt `SUPPORT_DEVICES_SRC` ở `C0` trỏ vào file 2 cột
`study_id`, `Support_Devices`.)*

In [ ]:
!gcloud auth application-default login --no-launch-browser
!gcloud auth application-default set-quota-project {PROJECT}

In [ ]:
if ADD_SUPPORT_DEVICES and SUPPORT_DEVICES_SRC is None:
    # gcloud tren Colab dat CLOUDSDK_CONFIG=/content/.config -> ADC KHONG nam o ~/.config/gcloud
    _cands = [os.environ.get('GOOGLE_APPLICATION_CREDENTIALS'),
              (os.path.join(os.environ['CLOUDSDK_CONFIG'], 'application_default_credentials.json')
               if os.environ.get('CLOUDSDK_CONFIG') else None),
              '/content/.config/application_default_credentials.json',
              os.path.expanduser('~/.config/gcloud/application_default_credentials.json')]
    _adc = next((p for p in _cands if p and os.path.exists(p)), None)
    if _adc:
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = _adc     # chan roi xuong GCE metadata
        print('ADC:', _adc)
    else:
        print('! khong thay file ADC o cac cho quen thuoc:')
        for p in _cands:
            if p:
                print('   -', p)
        print('  -> van thu google.auth.default(); cong ben duoi moi la cho quyet dinh')

    import google.auth
    _creds, _proj = google.auth.default()
    _kind = type(_creds).__module__
    print(f'credential: {type(_creds).__name__} ({_kind}) | quota project: {_proj}')
    assert 'compute_engine' not in _kind, (
        'dang dung credential cua GCE metadata -> cell gcloud chua chay thanh cong. '
        'Chay lai va dan ma xac thuc.')
    print('COÅNG · credential la ADC nguoi dung, khong phai GCE metadata  OK')
else:
    print('bo qua auth (dung SUPPORT_DEVICES_SRC hoac ADD_SUPPORT_DEVICES=False)')

In [ ]:
SD = None
if ADD_SUPPORT_DEVICES:
    if SUPPORT_DEVICES_SRC:
        _t = (pd.read_parquet(SUPPORT_DEVICES_SRC) if SUPPORT_DEVICES_SRC.endswith('.parquet')
              else pd.read_csv(SUPPORT_DEVICES_SRC))
        _t = _t.rename(columns={c: 'sd' for c in _t.columns if c != 'study_id'})
        print(f'nguon: {SUPPORT_DEVICES_SRC}  ({len(_t):,} hang)')
    else:
        from google.cloud import bigquery
        client = bigquery.Client(project=PROJECT, location='US')
        # doc ten cot TU SCHEMA, khong doan -- y het 01 C1
        _cols = [f.name for f in client.get_table(f'{CXR}.chexpert').schema]
        def _norm(c):
            return c.replace('_', ' ').strip().lower()
        _cand = [c for c in _cols if _norm(c) == 'support devices']
        assert len(_cand) == 1, f'khong tim thay dung 1 cot "support devices" trong {_cols}'
        SD_COL = _cand[0]
        print(f'cot trong BigQuery: `{SD_COL}`')
        _t = client.query(f'SELECT study_id, `{SD_COL}` AS sd FROM `{CXR}.chexpert`').to_dataframe()
        print(f'chexpert: {len(_t):,} hang')

    print('gia tri THO truoc U-policy:', dict(_t.sd.value_counts(dropna=False)))
    _v = _t.sd
    if U_POLICY == 'ones':
        _v = _v.replace(-1, 1)
    elif U_POLICY == 'zeros':
        _v = _v.replace(-1, 0)
    _t = _t.assign(sd=_v.fillna(0).astype('int8')).drop_duplicates('study_id')

    _m = pd.Series(_t.sd.to_numpy(), index=_t.study_id.to_numpy()).reindex(SID)
    # COÅNG: 01 assert khong co NaN sau merge ("has_chexpert lied"). Giu nguyen muc do nghiem.
    _miss = int(_m.isna().sum())
    print(f'COÅNG · phu het cohort: thieu {_miss} / {len(SID):,}  {"OK" if _miss == 0 else "FAIL"}')
    assert _miss == 0, (f'{_miss} study trong cohort khong co trong bang chexpert -> '
                        'nguon sai hoac cohort khong phai cohort_v2')
    SD = _m.to_numpy().astype('int8')
    assert set(np.unique(SD)) <= {0, 1}, f'gia tri la {set(np.unique(SD))}, phai la 0/1'

    print(f'\nSupport_Devices: prevalence cohort {SD.mean():.4f} | train {SD[IS_TRAIN].mean():.4f}'
          f'  ({int(SD.sum()):,} duong tinh)')
    print('  -> so voi 13 nhan kia day la nhan RAT PHO BIEN. Bao cao RIENG; dung gop vao')
    print('     macro chung roi so voi so cu 13 nhan -- hai con so do khong cung thang do.')
else:
    print('ADD_SUPPORT_DEVICES=False -> giu 13 nhan')

# ---------------- ma tran nhan + class_freq moi cho DB Loss ----------------
Y = np.column_stack([IDX0[l].to_numpy('int8') for l in LABELS13]
                    + ([SD] if ADD_SUPPORT_DEVICES else [])).astype('int8')
assert Y.shape == (N, C)
CLASS_FREQ = Y[IS_TRAIN].sum(0).astype('int64')
NUM_TRAIN  = int(IS_TRAIN.sum())
TRAIN_PREV = CLASS_FREQ / NUM_TRAIN
assert (CLASS_FREQ > 0).all(), 'co nhan khong xuat hien o train -> DB Loss se chia 0'
print(f'\nclass_freq moi ({C} nhan, train n={NUM_TRAIN:,}) -- '
      'class_freq.pkl cu chi co 13, DB Loss se mis-weight neu dung lai:')
print(pd.DataFrame({'label': LABELS, 'train_pos': CLASS_FREQ,
                    'train_prev': TRAIN_PREV.round(4)}).sort_values('train_prev').to_string(index=False))

## C2 · Manifest ảnh + CLS per-image

Xếp mọi ảnh frontal/lateral thành bảng **liên tục theo study**, theo đúng trật tự chủ của
`index.parquet`, trong mỗi study sắp theo `dicom_id` (trùng trật tự `05` đã dùng). Bố cục
này khiến pooling chỉ còn một `np.add.reduceat`.

*(Đọc ~1.06 GB từ Drive — cell chậm nhất, ~2–4 phút.)*

In [ ]:
t0   = time.time()
cidx = pd.read_parquet(os.path.join(CACHE_DIR, 'cxr_cls_index.parquet'))
vm   = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm   = vm[vm.view_group.isin(['frontal', 'lateral'])]

img = cidx.merge(vm[['dicom_id', 'study_id', 'view_group']], on='dicom_id', how='inner')
assert len(img) == len(cidx), f'{len(cidx)-len(img)} dicom trong cache khong co trong view_map'
sid_pos = pd.Series(np.arange(N), index=SID)
img = img[img.study_id.isin(sid_pos.index)].copy()
img['study_row'] = img.study_id.map(sid_pos).to_numpy()
img = img.sort_values(['study_row', 'dicom_id']).reset_index(drop=True)

srow     = img.study_row.to_numpy()
is_front = (img.view_group.to_numpy() == 'frontal')
n_img    = np.bincount(srow, minlength=N)
n_f      = np.bincount(srow[is_front],  minlength=N)
n_l      = np.bincount(srow[~is_front], minlength=N)
assert n_img.min() >= 1, 'co study khong con anh frontal/lateral nao'
OFFS = np.concatenate([[0], np.cumsum(n_img)])

E = np.load(os.path.join(CACHE_DIR, 'cxr_cls.npy'))[img.row.to_numpy()].astype(DTYPE)
assert len(E) == len(img) == int(n_img.sum())
print(f'CLS per-image theo manifest: {E.shape}  ({time.time()-t0:.0f}s)')
print(f'anh/study: tb {n_img.mean():.2f}, max {n_img.max()} | '
      f'frontal {int(is_front.sum()):,} lateral {int((~is_front).sum()):,}')

comp = np.select([(n_f > 0) & (n_l > 0), (n_f > 0) & (n_l == 0), (n_f == 0) & (n_l > 0)],
                 ['both', 'frontal_only', 'lateral_only'], 'none')
print('composition:', dict(pd.Series(comp).value_counts()))
assert (comp != 'none').all()

## C3 · Trọng số từng arm — **cổng tái lập `dataset_v3`**

Với arm `w_f`, ảnh thứ `i` của study `s` nhận

$$w_i=\frac{w_{\text{view}(i)}\,/\,n_{\text{view}(i)}(s)}{w_f\,[\exists F]+w_l\,[\exists L]}$$

Mean **trong** view trước, weight **giữa** view sau — để study có 3 ảnh frontal không lấn
át study có 1. Mẫu số là tái chuẩn hoá của `05`/`04b`: 51.6% study chỉ có frontal, nếu để
tổng có trọng số trần thì embedding của chúng bị co 0.7 lần và trôi khỏi manifold của nhóm
hai view.

Cả ba tỉ lệ `0.7/0.6/0.5` đều có `w_f, w_l > 0` nên mẫu số luôn dương — **không arm nào
trong ba arm này gặp vấn đề study lateral-only.** (Vấn đề đó chỉ phát sinh ở `w_f=1.0`,
tức arm `frontal`, chưa dựng ở đây.)

### Cổng

Dựng lại `0.7/0.3` từ cache per-image, chuẩn hoá bằng stats của `05`, đối chiếu với khối
`x_cxr` **đã nằm sẵn trong `dataset_v3`**. Khớp thì *thứ tự manifest*, *công thức trọng số*
và *cách chuẩn hoá* đều đúng cùng lúc. Đây là cổng đáng tin nhất notebook — nếu nó trượt,
đừng đọc gì thêm, dừng lại.

In [ ]:
den_f = np.where(n_f > 0, n_f, 1).astype(DTYPE)
den_l = np.where(n_l > 0, n_l, 1).astype(DTYPE)
has_f = (n_f > 0).astype(DTYPE)
has_l = (n_l > 0).astype(DTYPE)

LAT_ONLY = (n_f == 0)      # 366 study tren cohort that -- chi arm frontal cham toi

def arm_weights(w_f):
    """[n_img] float32, tong = 1 tren TUNG study (ke ca study lateral-only)."""
    assert 0.0 < w_f <= 1.0, f'w_f={w_f} ngoai khoang (0, 1]'
    w_l  = 1.0 - w_f
    norm = w_f * has_f + w_l * has_l
    per  = np.where(is_front, w_f / den_f[srow], w_l / den_l[srow]).astype(DTYPE)
    if w_f >= 1.0 - 1e-9:
        # w_f=1 => mau so = 0 o study lateral-only. Fallback: dung lateral mean ("dung cai
        # dang co") de N GIU NGUYEN giong 3 arm kia -- neu loai chung thi arm nay duoc cham
        # tren tap study khac va Delta khong con quy ve pooling. Cot frontal_is_fallback
        # trong index.parquet danh dau chinh xac chung; muon theo quy uoc 04b thi loc o
        # DOWNSTREAM, tren CA 4 arm.
        per  = np.where(LAT_ONLY[srow] & ~is_front, (1.0 / den_l[srow]).astype(DTYPE), per)
        norm = np.where(LAT_ONLY, 1.0, norm)
    assert norm.min() > 0, 'mau so tai chuan hoa = 0 -> co study khong duoc arm nay phu'
    return (per / norm[srow]).astype(DTYPE)

def pool(w):
    return np.add.reduceat(E * w[:, None], OFFS[:-1], axis=0).astype(DTYPE)

def standardize(X):
    mu = X[IS_TRAIN].mean(0).astype(DTYPE)
    sd = (X[IS_TRAIN].std(0) + 1e-6).astype(DTYPE)
    return ((X - mu) / sd).astype(DTYPE), mu, sd

for _a, _wf in ARMS.items():
    _s = np.abs(np.add.reduceat(arm_weights(_wf), OFFS[:-1]) - 1.0).max()
    print(f'COÅNG · sum(w)=1  {_a:16s} w_f={_wf}  max|sum-1| {_s:.2e}  '
          f'{"OK" if _s <= 1e-5 else "FAIL"}')
    assert _s <= 1e-5, f'{_a}: trong so khong tong 1 -> arm nay dang doi scale study'

# ---------------- COÅNG: 0.7/0.3 tai lap khoi x_cxr cua dataset_v3 ----------------
_x73 = pool(arm_weights(0.7))
_stp = os.path.join(CACHE_DIR, 'x_cxr_train_stats.npz')
if SPEC0['x_cxr'].get('standardized', True):
    assert os.path.exists(_stp), f'{_stp} khong co, nhung dataset_v3 la ban da chuan hoa'
    _st  = np.load(_stp)
    _x73 = ((_x73 - _st['mean']) / _st['std']).astype(DTYPE)
_e = float(np.abs(_x73 - XREF).max())
print(f'\nCOÅNG · 0.7/0.3 tai lap x_cxr trong dataset_v3: max_abs_diff {_e:.2e}  '
      f'{"OK" if _e <= 2e-3 else "FAIL"}')
assert _e <= 2e-3, ('khoi x_cxr dung lai KHONG khop dataset_v3 -> sai thu tu manifest, sai '
                    'cong thuc trong so, hoac sai cach chuan hoa. DUNG LAI, dung ghi gi ca.')

# ---------------- COÅNG: arm frontal -- fallback dung cho dung 366 study, khong hon ----------------
if any(w >= 1.0 - 1e-9 for w in ARMS.values()):
    _xf = pool(arm_weights(1.0))
    _Fs = np.zeros((N, DX), 'float64'); _Ls = np.zeros((N, DX), 'float64')
    np.add.at(_Fs, srow[is_front],  E[is_front])
    np.add.at(_Ls, srow[~is_front], E[~is_front])
    _Fm = (_Fs / den_f[:, None]).astype(DTYPE)      # trung binh frontal
    _Lm = (_Ls / den_l[:, None]).astype(DTYPE)      # trung binh lateral
    _exp = np.where(LAT_ONLY[:, None], _Lm, _Fm)
    _e1 = float(np.abs(_xf[~LAT_ONLY] - _Fm[~LAT_ONLY]).max())
    _e2 = float(np.abs(_xf[LAT_ONLY] - _Lm[LAT_ONLY]).max()) if LAT_ONLY.any() else 0.0
    print(f'\nCOÅNG · frontal, study CO frontal  -> trung binh frontal: {_e1:.2e}  '
          f'{"OK" if _e1 <= 1e-4 else "FAIL"}')
    print(f'COÅNG · frontal, study lateral-only -> trung binh lateral: {_e2:.2e}  '
          f'{"OK" if _e2 <= 1e-4 else "FAIL"}  ({int(LAT_ONLY.sum()):,} study)')
    assert max(_e1, _e2) <= 1e-4, 'logic fallback cua arm frontal SAI'
    assert float(np.abs(_xf - _exp).max()) <= 1e-4
    del _xf, _Fs, _Ls, _Fm, _Lm, _exp

# stats per-arm cua 0.7/0.3 phai TRUNG stats cua 05 (vi do dung la cach 05 pool)
if os.path.exists(_stp):
    _, _mu, _sd = standardize(pool(arm_weights(0.7)))
    _st = np.load(_stp)
    _d  = max(float(np.abs(_mu - _st['mean']).max()), float(np.abs(_sd - _st['std']).max()))
    print(f'COÅNG · stats per-arm cua 0.7/0.3 == stats cua 05: max_abs_diff {_d:.2e}  '
          f'{"OK" if _d <= 1e-3 else "FAIL"}')
    assert _d <= 1e-3
del _x73, XREF

## C4 · Ghi 4 thư mục vào `planB_out`

Mỗi arm: pool → chuẩn hoá bằng stats train-only **của chính nó** → ghép với khối
`micro ⊕ ehr` → ghi. `index.parquet` sao chép nguyên (thứ tự hàng phải giống hệt mọi arm),
`columns.json` giữ nguyên spec của `07` và chỉ đổi phần mô tả `x_cxr`.

*(CPU. ~4–7 phút/arm, gần như toàn bộ là ghi ~1.8 GB xuống Drive.)*

In [ ]:
def write_arm(dirname, w_f):
    d = os.path.join(DATA_DIR, dirname)
    os.makedirs(d, exist_ok=True)

    X, mu, sd = standardize(pool(arm_weights(w_f)))
    assert not np.isnan(X).any(), f'{dirname}: NaN trong x_cxr'
    np.savez(os.path.join(d, 'x_cxr_train_stats.npz'), mean=mu, std=sd)

    idx = IDX0.copy()
    is_frontal_arm = w_f >= 1.0 - 1e-9
    if ADD_SUPPORT_DEVICES:
        idx['Support_Devices'] = SD                # nhan thu 14, sau 13 nhan cu
    if is_frontal_arm:
        # danh dau chinh xac study nao dung lateral mean thay vi frontal
        idx['frontal_is_fallback'] = LAT_ONLY
    assert [c for c in idx.columns if c in LABELS] == LABELS, 'thu tu nhan trong index lech LABELS'
    idx.to_parquet(os.path.join(d, 'index.parquet'), index=False)

    # class_freq cho DB Loss -- cai cu trong cohort_v2/ chi co 13 nhan
    with open(os.path.join(d, 'class_freq.pkl'), 'wb') as fh:
        pickle.dump(dict(labels=LABELS, class_freq=CLASS_FREQ.tolist(), num_train=NUM_TRAIN), fh)

    spec = json.loads(json.dumps(SPEC0))                       # deep copy
    spec['labels'] = LABELS                                    # 14 nhan, KHONG phai 13 cua SPEC0
    spec['n_labels'] = C
    spec['label_note'] = (
        'Support_Devices duoc lay lai tu physionet-data.mimic_cxr_jpg.chexpert voi U-Ones '
        '(-1->1, NaN->0), dung quy uoc cua 01. 01 co chu y bo no vi "not a pathology". '
        'No la confound da biet (ICU/benh nang) -- bao cao RIENG, dung gop vao macro roi so '
        'voi so cu 13 nhan.') if ADD_SUPPORT_DEVICES else '13 nhan nhu dataset_v3'
    spec['files'] = dict(
        index='index.parquet (study_id, split, setting, row, %d nhan)' % C,
        class_freq='class_freq.pkl (labels/class_freq/num_train cho %d nhan)' % C,
        cohort=('cohort_study_level.parquet (bang cohort day du'
                + (' + x_cxr cua arm nay)' if COHORT_PARQUET_WITH_XCXR else ')')
                if WRITE_COHORT_PARQUET else 'khong ghi'))
    spec['arm'] = dict(dir=dirname, w_f=float(w_f), w_l=float(1.0 - w_f),
                       pooling=('chi frontal (mean trong view)' if is_frontal_arm else
                                f'mean trong view, roi w_f={w_f}/w_l={1.0-w_f:.1f} '
                                'voi mau so tai chuan hoa'))
    if is_frontal_arm:
        spec['arm'].update(
            n_lateral_only=int(LAT_ONLY.sum()),
            lateral_only_policy='fallback sang lateral mean, giu N giong 3 arm kia',
            fallback_column='frontal_is_fallback (trong index.parquet cua RIENG arm nay)',
            note_04b=('quy uoc 04b la LOAI han cac study do. Muon theo no thi loc '
                      'frontal_is_fallback o DOWNSTREAM, tren CA 4 arm -- chi loc o arm nay '
                      'se khien no duoc cham tren tap study khac va Delta khong con quy ve pooling'))
    spec['x_cxr'] = dict(dim=DX, source=f'05 cxr_cls.npy (per-image), pooled w_f={w_f}',
                         standardized=True,
                         stats='x_cxr_train_stats.npz (train-only, PER-ARM)')
    spec['derived_from'] = ('dataset_v3 (07): khoi micro+ehr bit-identical, '
                            'chi khoi x_cxr duoc dung lai')
    spec['variants'] = {v: SPEC0['variants'][v] for v in VARIANTS}
    json.dump(spec, open(os.path.join(d, 'columns.json'), 'w'), indent=1)

    for v in VARIANTS:
        cols = SPEC0['variants'][v]['columns']
        p_npy = os.path.join(d, SPEC0['variants'][v]['file'])
        if os.path.exists(p_npy) and not FORCE_REWRITE_NPY:
            # .npy KHONG phu thuoc nhan -> them nhan 14 khong can ghi lai 1.8 GB/arm
            _sh = np.load(p_npy, mmap_mode='r').shape
            assert _sh == (N, len(cols)), f'{p_npy} shape {_sh} != {(N, len(cols))}'
            print(f'    {os.path.basename(p_npy):22s} da co, BO QUA  {_sh}')
            continue
        Xv = np.concatenate([STATIC[v], X], 1).astype(DTYPE)
        assert Xv.shape == (N, len(cols)), (v, Xv.shape, len(cols))
        np.save(p_npy, Xv)
        print(f'    {os.path.basename(p_npy):22s} {Xv.shape}  {Xv.nbytes/1e9:.2f} GB')
        if WRITE_CSV_SAMPLE:
            n  = min(WRITE_CSV_SAMPLE, N)
            df = pd.concat([idx.head(n).reset_index(drop=True),
                            pd.DataFrame(Xv[:n], columns=cols)], axis=1)
            df.to_csv(os.path.join(d, f'{v}_sample{n}.csv'), index=False, float_format='%.6g')
            del df
        del Xv

    # ---- cohort_study_level.parquet cua RIENG arm nay ----
    if WRITE_COHORT_PARQUET:
        coh = COHORT.copy()
        if ADD_SUPPORT_DEVICES:
            coh['Support_Devices'] = SD
        if is_frontal_arm:
            coh['frontal_is_fallback'] = LAT_ONLY
        if COHORT_PARQUET_WITH_XCXR:
            assert not any(c.startswith('x_cxr_') for c in coh.columns), 'trung ten cot x_cxr_*'
            coh = pd.concat([coh.reset_index(drop=True),
                             pd.DataFrame(X, columns=[f'x_cxr_{i}' for i in range(DX)])], axis=1)
        p_coh = os.path.join(d, 'cohort_study_level.parquet')
        coh.to_parquet(p_coh, compression='snappy', index=False)
        print(f'    {"cohort_study_level.parquet":22s} {coh.shape}  '
              f'{os.path.getsize(p_coh)/1e6:.0f} MB')
        del coh
    print(f'    x_cxr sau chuan hoa: mean {X.mean():+.5f}  std {X.std():.5f}'
          + (f'  | {int(LAT_ONLY.sum()):,} study fallback sang lateral mean' if is_frontal_arm else ''))
    del X

t0 = time.time()
for dirname, w_f in ARMS.items():
    print(f'\n--- {dirname}  (w_f={w_f} / w_l={1.0-w_f:.1f}) ---')
    write_arm(dirname, w_f)
print(f'\nghi xong {len(ARMS)} arm trong {(time.time()-t0)/60:.1f} phut')

## C5 · Nghiệm thu — nạp lại từ đĩa như bên nhận sẽ nạp

Bốn thứ chỉ kiểm được **sau khi** đã ghi:

1. **Khối non-image bit-identical giữa mọi arm.** Không thì Δ giữa các arm lẫn cả thứ khác
   ngoài pooling, và cả bộ dataset vô nghĩa.
2. **`x_cxr` thực sự khác nhau giữa các arm.** Nghe hiển nhiên, nhưng một lỗi copy-paste
   ghi cùng một mảng ra 4 thư mục sẽ trông hoàn toàn bình thường.
3. **`dataset_v3_73` ≈ `dataset_v3`** trên toàn ma trận, không chỉ khối `x_cxr`.
4. **`NaN` chỉ ở khối EHR**, đúng thiết kế `07`; micro và `x_cxr` không bao giờ.

In [ ]:
v0    = VARIANTS[0]
DIRS  = [('dataset_v3', SRC_DS)] + [(d, os.path.join(DATA_DIR, d)) for d in ARMS]
ref_static, xs = None, {}

print('=== tung thu muc ===')
for name, d in DIRS:
    sp = json.load(open(os.path.join(d, 'columns.json')))
    ix = pd.read_parquet(os.path.join(d, 'index.parquet'))
    X  = np.load(os.path.join(d, sp['variants'][v0]['file']), mmap_mode='r')
    assert X.shape == (N, len(sp['variants'][v0]['columns'])), (name, X.shape)
    assert (ix.study_id.to_numpy() == SID).all(), f'{name}: study_id lech trat tu'
    assert (ix.row.to_numpy() == np.arange(N)).all(), f'{name}: row lech'
    exp_lab = LABELS13 if name == 'dataset_v3' else LABELS      # nguon 13, arm 14
    assert sp['labels'] == exp_lab, f'{name}: nhan lech ({len(sp["labels"])} vs {len(exp_lab)})'
    assert len(sp['labels']) != 40, f'{name}: 40 nhan -> nham nhanh CXR-LT'
    for _l in exp_lab:
        assert _l in ix.columns, f'{name}: index.parquet thieu cot nhan {_l}'
    if name != 'dataset_v3' and ADD_SUPPORT_DEVICES:
        assert (ix.Support_Devices.to_numpy() == SD).all(), f'{name}: Support_Devices lech'
        _cf = pickle.load(open(os.path.join(d, 'class_freq.pkl'), 'rb'))
        assert _cf['labels'] == LABELS and len(_cf['class_freq']) == C, f'{name}: class_freq.pkl sai'
        if WRITE_COHORT_PARQUET:
            _cp = os.path.join(d, 'cohort_study_level.parquet')
            assert os.path.exists(_cp), f'{name}: thieu cohort_study_level.parquet'
    st = np.ascontiguousarray(X[:, :-DX]); xc = np.ascontiguousarray(X[:, -DX:])
    assert not np.isnan(xc).any(), f'{name}: NaN trong x_cxr'
    if ref_static is None:
        ref_static = st
    else:
        e = float(np.nanmax(np.abs(st - ref_static)))
        assert e == 0.0, f'{name}: khoi non-image KHAC cac arm khac ({e:.2e}) -> Delta se lan tap'
    xs[name] = xc[:5000].copy()
    print(f'  {name:16s} {tuple(X.shape)}  x_cxr mean {xc.mean():+.5f} std {xc.std():.5f}   OK')
    del X, st, xc
print('  -> khoi micro+ehr bit-identical o moi thu muc')

print('\n=== x_cxr phai KHAC nhau giua cac arm ===')
nm = list(xs)
for i in range(len(nm)):
    for j in range(i + 1, len(nm)):
        e = float(np.abs(xs[nm[i]] - xs[nm[j]]).max())
        flag = 'OK' if e > 1e-6 else 'TRUNG NHAU'
        if nm[i] == 'dataset_v3' and nm[j] == 'dataset_v3_73':
            print(f'  {nm[i]:16s} vs {nm[j]:16s} max_diff {e:.2e}  '
                  f'{"OK (phai gan bang nhau)" if e <= 2e-3 else "FAIL"}')
            assert e <= 2e-3, 'dataset_v3_73 phai tai lap dataset_v3'
        else:
            print(f'  {nm[i]:16s} vs {nm[j]:16s} max_diff {e:.2e}  {flag}')
            assert e > 1e-6, f'{nm[i]} va {nm[j]} co x_cxr GIONG HET -> loi ghi'

print('\n=== NaN chi o khoi EHR ===')
# tach khoi theo TEN cot, khong hardcode be rong -- 07 doi vocab thi be rong doi theo
_cols = SPEC0['variants'][v0]['columns']
_pref = tuple(f'{k}__' for k in SPEC0.get('micro', {}).get('blocks', {}))
_is_micro = np.array([c.startswith(_pref) for c in _cols]) if _pref else np.zeros(len(_cols), bool)
_is_x     = np.array([c.startswith('x_cxr_') for c in _cols])
_is_ehr   = ~_is_micro & ~_is_x
X = np.load(os.path.join(DATA_DIR, list(ARMS)[0], SPEC0['variants'][v0]['file']), mmap_mode='r')
if _pref:
    assert not np.isnan(np.ascontiguousarray(X[:, _is_micro])).any(), 'NaN trong khoi micro -> loi that su'
    print(f'  micro ({int(_is_micro.sum())} cot): khong NaN')
else:
    print('  ! columns.json khong khai bao micro.blocks -> bo qua kiem micro')
_r = float(np.isnan(np.ascontiguousarray(X[:, _is_ehr])).mean()) if _is_ehr.any() else 0.0
print(f'  ehr   ({int(_is_ehr.sum())} cot): {_r:.1%} NaN (dung thiet ke 07)')
print(f'  x_cxr ({int(_is_x.sum())} cot): khong NaN')
assert int(_is_x.sum()) == DX
del X
print('\nTAT CA CONG DA QUA.')

## C6 · Liệt kê + cách nạp

In [ ]:
print('=== da ghi vao', DATA_DIR, '===')
tot = 0
for name in ARMS:
    d  = os.path.join(DATA_DIR, name)
    sz = sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d))
    tot += sz
    print(f'\n{name}/   ({sz/1e9:.2f} GB)')
    for f in sorted(os.listdir(d)):
        print(f'    {f:32s} {os.path.getsize(os.path.join(d, f))/1e6:9.1f} MB')
print(f'\nTONG GHI MOI: {tot/1e9:.2f} GB   (dataset_v3 co san khong bi dung toi)')

print(r"""
--- cach nap ---

import numpy as np, pandas as pd, json

D    = 'dataset_v3_64'         # _73 | _64 | _55 | _frontal   <- doi arm o day
idx  = pd.read_parquet(f'{D}/index.parquet')
spec = json.load(open(f'{D}/columns.json'))

# v2_embedding: micro dang embedding (829 cot) -- dung cho mang no-ron
X  = np.load(f'{D}/v2_embedding.npy')
# v1_onehot   : micro dang one-hot  (1272 cot) -- dung cho cay (XGBoost doc NaN truc tiep)
# X = np.load(f'{D}/v1_onehot.npy')

tr, va, te = (idx.split == s for s in ('train', 'val', 'test'))
Xtr, ytr = X[idx.row[tr]], idx.loc[tr, spec['labels']].to_numpy()
Xva, yva = X[idx.row[va]], idx.loc[va, spec['labels']].to_numpy()

d = spec['x_cxr']['dim']                  # 768
x_cxr, static = X[:, -d:], X[:, :-d]      # x_cxr khac giua cac arm; static giong het

# ablation: lap qua 3 arm, moi thu khac giu nguyen
for D in ['dataset_v3_73', 'dataset_v3_64', 'dataset_v3_55', 'dataset_v3_frontal']:
    X = np.load(f'{D}/v2_embedding.npy')
    ...                                    # >= 3 seed, so Delta voi sd giua cac seed
""")

## Còn lại: arm `attention`

Chưa dựng ở đây, và nó **không thể cùng dạng** với ba arm trên: trọng số attention do model
sinh ra từ chính feature lúc forward, nên không tồn tại `x_cxr` tĩnh — không có cách nào
đóng gói nó thành `[211130, 768]` mà vẫn còn là attention.

Thư mục cho nó sẽ phải chứa nguyên liệu để model tự pool:

| file | shape |
|---|---|
| `x_cxr_per_image.npy` | `[346578, 768]` — CLS **từng ảnh**, thứ tự `image_manifest.parquet` |
| `ragged.npz` | `pad_idx [N,K]`, `pad_mask`, `pad_is_frontal` |
| `static.npy` | `[211130, 61]` — khối `micro ⊕ ehr`, giống hệt ba arm trên |

Model pool `[B,K,768] → [B,768]` rồi `hstack` với `static` là ra **đúng layout `[N, 829]`**
của bốn arm này, nên head phía sau dùng chung được, không phải viết hai đường.

Toàn bộ máy móc đó đã có sẵn trong [`27b_multiview_input_export.ipynb`](
27b_multiview_input_export.ipynb) (`ragged.npz` + `mv_pool.py` với `MultiViewPool`
gated-attention) — khi cần chỉ là ghép nó với `static.npy` ở đây.

**Một điểm nên giữ khi dựng:** cho score attention đọc cờ frontal/lateral, và init trọng số
score bằng 0. Có cờ thì `attention` biểu diễn được cả ba arm tỉ lệ cố định (chỉ cần học một
hằng số theo view) ⇒ siêu tập đúng nghĩa, so sánh mới có ý nghĩa. Init 0 thì nó khởi động ở
trung bình đều, tức phải tự kiếm mọi sai lệch — và hiệu so với arm trung-bình-đều đọc được
như phần attention thực học, tách khỏi phần chỉ do thêm tham số.